# 5a. Visualising the source estimates (without pyvista)

Usually, within python scripts and MNE python we plot the source estimates with `stc.plot()`, which needs the pyvista/VTK 3D
backend (`mne.viz.set_3d_backend`) and rendering which on some machines is difficult to use - like on our Binder/JupyterHub server.
 
Therefore we are here using **only matplotlib** via `nilearn.plotting.plot_surf_stat_map` / `plot_surf_roi` with `engine="matplotlib"`. Those functions render the cortical mesh with matplotlib's built-in 3D toolkit (`mplot3d`) instead
of pyvista/VTK, so there is no dependency on pyvista, VTK, trame or a GPU at all. 

<div class="alert alert-success">
    <b>Learning Objectives</b>:
     <ul>
      <li>Visualizing the computed reconstructed activity.</li>
      <li>Interpreting our results: do we see a meaningful contrast for our condition perceived vs. unperceived?</li>
    </ul>
</div>

In [ ]:
# imports - no pyvista, no mne.viz.set_3d_backend
import os

import numpy as np
import matplotlib.pyplot as plt
from nibabel.freesurfer.io import read_morph_data

import mne
from nilearn.plotting import plot_surf_stat_map, plot_surf_roi

print("MNE:", mne.__version__)

# data paths 
subID = 24
data_path = "data"
subject_path = os.path.join(f"sub-0{subID}", "ses-mecha", "eeg")
base = os.path.join("..", data_path, subject_path, f"sub-0{subID}_ses-mecha_task-NT_")

# fsaverage: downloads it on first use, otherwise just returns the path
fs_dir = mne.datasets.fetch_fsaverage()
subjects_dir = os.path.dirname(fs_dir)
print(subjects_dir)

## 5.1 Morphing to fsaverage

We morph the source estimate onto *fsaverage*, which we downloaded previously, and with the precomputed morph.

In [ ]:
# load the source estimate and the precomputed morph
stc = mne.read_source_estimate(base + "MNE-all")
morph = mne.read_source_morph(base + "morph.h5")

# apply the morph: individual source space -> fsaverage
stc_fs = morph.apply(stc)
print(stc_fs)

`stc_fs` lives on the fsaverage *source space*: 10242 vertices per hemisphere (ico-5). The
fsaverage *surface mesh* we are about to paint it on has 163842. 

Morphing fsaverage onto itself with `spacing=None` spreads the values over every vertex of
the mesh.

In [ ]:
morph_surface = mne.compute_source_morph(
    stc_fs,
    subject_from="fsaverage",
    subject_to="fsaverage",
    spacing=None,  # None = keep every vertex of the target surface
    smooth=10, # the number of smoothing steps.
    subjects_dir=subjects_dir,
)
print(morph_surface)

## Helper functions

.. You might skip this (but make sure to execute the code cell) ...


Two things `stc.plot()` normally does for us under the hood, but since we circumvent the 3D plotting here we need:

- read the cortical surface mesh (coordinates + triangles) - here via `mne.read_surface`,
  a pure Python/nibabel reader, no VTK involved.
- hand one hemisphere's values for one time point to the renderer. After the morph above
  `stc_surf` has exactly one value per surface vertex, so `lh_data` / `rh_data` line up with
  the mesh row by row.

In [ ]:
def load_hemi_mesh(hemi):
    """Coordinates, triangles and sulcal-depth background map for one fsaverage hemisphere."""
    surf_dir = os.path.join(subjects_dir, "fsaverage", "surf")
    coords, faces = mne.read_surface(os.path.join(surf_dir, f"{hemi}.inflated"))
    sulc = read_morph_data(os.path.join(surf_dir, f"{hemi}.sulc"))
    return coords, faces, sulc


def surface_values(stc, time):
    """Both hemispheres of a source estimate at one time point, morphed to the full fsaverage surface."""
    # morph only this time point: all time points at full resolution take ~0.5 GB per source estimate
    idx = stc.time_as_index(time)[0]
    values = morph_surface.morph_mat @ stc.data[:, idx]
    n_lh = len(morph_surface.vertices_to[0])
    return {"lh": values[:n_lh], "rh": values[n_lh:]}


# cache the two hemisphere meshes - reused by every plot below
meshes = {hemi: load_hemi_mesh(hemi) for hemi in ("lh", "rh")}

# the two must match, otherwise the values do not belong to the vertices they are drawn on
print("mesh vertices:", meshes["lh"][0].shape[0], "- morph vertices:", len(morph_surface.vertices_to[0]))

def plot_stat_map_both_hemis(stc, time, title, threshold, vmax):
    """Lateral view of both hemispheres for one time point, via nilearn's matplotlib engine.

    `threshold` and `vmax` are computed once by the caller and shared by both hemispheres,
    so the two brains are on the same colour scale and can be compared directly.

    The threshold is not optional. With `threshold=None`, nilearn keeps every vertex and
    draws the map fully opaque on top of `bg_map`, which buries the sulcal anatomy under a
    near-white overlay. `stc.plot()` hides this from us by thresholding at the 93rd
    percentile by default.
    """
    values = surface_values(stc, time)
    fig = plt.figure(figsize=(10, 4))
    brain_axes = []
    for i, hemi in enumerate(("lh", "rh")):
        coords, faces, sulc = meshes[hemi]
        ax = fig.add_subplot(1, 2, i + 1, projection="3d")
        plot_surf_stat_map(
            [coords, faces], values[hemi], bg_map=sulc,
            hemi="left" if hemi == "lh" else "right", view="lateral",
            engine="matplotlib", axes=ax,
            threshold=threshold, vmax=vmax, symmetric_cbar=True,
            colorbar=(hemi == "rh"),  # one colour bar, since both hemispheres share the scale
        )
        ax.set_title(hemi)
        brain_axes.append(ax)

    # nilearn puts its colour bar flush against the rh axes (pad=0), and subplots_adjust
    # would reset the axes underneath it again - so place brains and colour bar by hand
    cax = [a for a in fig.axes if a not in brain_axes][0]
    brain_axes[0].set_position([0.00, 0.02, 0.42, 0.84])
    brain_axes[1].set_position([0.40, 0.02, 0.42, 0.84])
    cax.set_position([0.88, 0.25, 0.02, 0.50])
    fig.suptitle(title)
    return fig

## 5.2. Plotting the contrast on the brain

The perceived-minus-unperceived contrast, morphed to fsaverage, plotted on the lateral view of the brain.

In [ ]:
stc_difference = mne.read_source_estimate(base + "MNE-difference")
stc_difference_fs = morph.apply(stc_difference)

diff_vertex, diff_time = stc_difference_fs.get_peak(tmin=0.05, tmax=0.3)
print(f"largest difference at {diff_time * 1000:.0f} ms")

diff_abs_values = np.abs(np.concatenate(list(surface_values(stc_difference_fs, diff_time).values())))

plot_stat_map_both_hemis(
    stc_difference_fs, diff_time, f"perceived - unperceived, {diff_time * 1000:.0f} ms",
    threshold=np.percentile(diff_abs_values, 97),
    vmax=np.percentile(diff_abs_values, 99.95),
)
plt.show()

<div class="alert alert-warning">
    <b>Exercise</b>:
    <ul>
        <li>In which brain areas do you see activation?</li>
        <li>What does red and blue activation mean?</li>
    </ul>
</div>

### 5.3 The Colour scale

The color scale of the plots determine how many sources are shown and how dark they are plotted. 
Example: setting the threshold to the 97th percentile, values below become transparent, and the colour scale saturates at `vmax`, so at the 99.95th percentile in this example. 

<div class="alert alert-warning">
    <b>Exercise</b>:
    Experiment with different values of the "threshold" and "vmax" parameter. What do you observe?
</div>

In [ ]:
abs_values = np.abs(stc_fs.data)
threshold = np.percentile(abs_values, 95)
vmax = np.percentile(abs_values, 99.95)

plot_stat_map_both_hemis(
    stc_difference_fs, diff_time, f"peak activation, {diff_time * 1000:.0f} ms (thresholded)",
    threshold=threshold, vmax=vmax,
)
plt.show()

## 5.4 Time course of a single source

We have a look at a single vertex activity. Play around with the vertex number to show different reconstructed traces.

In [ ]:
peak_idx, peak_time = stc_difference_fs.get_peak(tmin=0.05, tmax=0.3, vert_as_index=True)
n_lh = len(stc_fs.vertices[0])
hemi = "lh" if peak_idx < n_lh else "rh"
print(f"strongest source: row {peak_idx} ({hemi}), peaking at {peak_time * 1000:.0f} ms")

plt.plot(stc_difference_fs.times[100:], stc_difference_fs.data[peak_idx][100:])
plt.axvline(0, color="k", linestyle="--")
plt.axvline(peak_time, color="r", linestyle=":")
plt.xlabel("time (s)")
plt.ylabel("amplitude (Am)")
plt.title(f"peak source ({hemi})")
plt.show()

## 5.5 Anatomical parcels

Instead of single vertices we can average within anatomical regions. `aparc` is the
Desikan-Killiany parcellation that comes with FreeSurfer: 68 cortical labels, defined on
fsaverage and therefore directly usable on our morphed estimate.

Here, we want to compute the activity of the primary somatosensory cortex.
How to average the vertices can be chosen via the argument `mode`.
For example: `mode="mean_flip"`, which flips the sign of vertices whose surface normal points the other way
before averaging. 

Remember: There is still source leakage, so a label time course is **not** only "the activity
of that region". 

<div class="alert alert-warning">
    <b>Exercise</b>:
     <ul>
      <li>In which cases is it reasonable to choose `mean_flip` or `pca_flip` over `mean` as averaging option?</li>
      <li>Which option would you chose for the primary somatosensory cortex? (Tipp: look up the geometry of S1 and the pattern of the source reconstructed activity)</li>
    </ul>
</div>

In [ ]:
# the parcellation and the source space it is defined on
labels = mne.read_labels_from_annot("fsaverage", parc="aparc", subjects_dir=subjects_dir)
src_fs = mne.read_source_spaces(
    os.path.join(subjects_dir, "fsaverage", "bem", "fsaverage-ico-5-src.fif")
)

# a few regions that are plausible for a tactile detection task
roi_names = ["postcentral-lh", "postcentral-rh"]
rois = [label for label in labels if label.name in roi_names]

label_tc = mne.extract_label_time_course(stc_difference_fs, rois, src_fs, mode="pca_flip")
print("label time courses:", label_tc.shape)

In [ ]:
for name, time_course in zip([label.name for label in rois], label_tc):
    plt.plot(stc_difference_fs.times[100:], time_course[100:], label=name)
plt.axvline(0, color="k", linestyle="--")
plt.xlabel("time (s)")
plt.ylabel("amplitude (Am)")
plt.legend()
plt.title("label time courses, perceived - unperceived")
plt.show()

In [ ]:
# visualize where these labels sit on the brain
fig = plt.figure(figsize=(9, 4))
for i, hemi in enumerate(("lh", "rh")):
    coords, faces, sulc = meshes[hemi]
    # NaN, not 0: plot_surf_roi has no notion of a background value and would paint every
    # unlabelled vertex with the first colour of the colormap. NaN it masks out.
    roi_map = np.full(coords.shape[0], np.nan)
    for label_idx, label in enumerate(rois, start=1):
        if label.hemi == hemi:
            roi_map[label.vertices] = label_idx

    ax = fig.add_subplot(1, 2, i + 1, projection="3d")
    plot_surf_roi(
        [coords, faces], roi_map, bg_map=sulc,
        hemi="left" if hemi == "lh" else "right", view="lateral",
        engine="matplotlib", axes=ax, cmap="tab10", vmin=0, vmax=len(rois),
        colorbar=False,
    )
    ax.set_title(hemi)
fig.suptitle("ROI locations")
fig.subplots_adjust(wspace=0.0)

# precuneus sits on the medial wall, so it does not show up in a lateral view
print("ROIs, in colormap order:")
for label_idx, label in enumerate(rois, start=1):
    print(f"  {label_idx}: {label.name}")

## 5.6 Comparing LCMV and MNE inverse solution

In [ ]:
# load the LCMV contrast (perceived - unperceived) from the beamformer solution notebook
stc_lcmv_difference = mne.read_source_estimate(base + "lcmv-difference")
stc_lcmv_difference_fs = morph.apply(stc_lcmv_difference)

lcmv_vertex, lcmv_time = stc_lcmv_difference_fs.get_peak(tmin=0.05, tmax=0.3)
print(f"largest LCMV difference at {lcmv_time * 1000:.0f} ms (MNE: {diff_time * 1000:.0f} ms)")

# plot at the same time point as the MNE contrast above, so the two maps can be compared.
# LCMV (unit-noise-gain) and MNE values have different units, so each gets its own colour scale.
lcmv_abs_values = np.abs(np.concatenate(list(surface_values(stc_lcmv_difference_fs, diff_time).values())))

plot_stat_map_both_hemis(
    stc_lcmv_difference_fs, diff_time, f"LCMV: perceived - unperceived, {diff_time * 1000:.0f} ms",
    threshold=np.percentile(lcmv_abs_values, 97),
    vmax=np.percentile(lcmv_abs_values, 99.95),
)
plt.show()

<div class="alert alert-warning">
    <b>Exercise</b>:
    <ul>
        <li>What differences do you spot when comparing the solutions from the LCMV and MNE beamformer?</li>
        <li>What might be the reasons for different outcomes?</li>
    </ul>
</div>

## 5.7 Source activation at the N140 component
According to previous literature, the 140 component reflects conscious perception of somatosensory stimuli (). Now, let's plot the activated sources at that timepoint for both of our methods.

In [ ]:
n140_time = 0.14  # s

# one plot per inverse method; each gets its own colour scale, since MNE is in Am and LCMV in pseudo-Z
for name, stc_contrast in [("MNE", stc_difference_fs), ("LCMV", stc_lcmv_difference_fs)]:
    abs_values = np.abs(np.concatenate(list(surface_values(stc_contrast, n140_time).values())))
    plot_stat_map_both_hemis(
        stc_contrast, n140_time, f"{name}: perceived - unperceived, {n140_time * 1000:.0f} ms",
        threshold=np.percentile(abs_values, 95),
        vmax=np.percentile(abs_values, 99.95),
    )
    plt.show()